# "THE POUR IS RIGHT" - Wine Pricer Capstone

My own version of Ed's Week 6-8 project: a model that predicts how much a bottle of wine costs from its review.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 1: Data Curation

Today we scrub the dataset and curate it down to 22,000 wines.

The dataset is here:  
https://huggingface.co/datasets/spawn99/wine-reviews

This notebook follows `week6/day1.ipynb`. Each section says what Ed did at the same point, and what is different for wine.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from datasets import load_dataset, concatenate_datasets
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from collections import Counter
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse

In [ ]:
# Log in to HuggingFace and find out my username, which we need at the end to push the dataset

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=False)

username = HfApi().whoami()["name"]
print(f"Logged in as {username}")

## Load our dataset

Ed loaded one Amazon category at a time, and needed a parallel loader because there were millions of products.

The wine dataset is far smaller: 3 splits with 280,901 reviews in total. We join the 3 splits back into one, because we will make our own train / validation / test split at the end.

In [ ]:
splits = load_dataset("spawn99/wine-reviews")
splits

In [ ]:
dataset = concatenate_datasets([splits["train"], splits["validation"], splits["test"]])
print(f"Number of wine reviews: {len(dataset):,}")

In [ ]:
# Investigate a particular datapoint

dataset[6]

In [ ]:
# What's the most expensive wine?

max_price = 0
max_datapoint = None

for datapoint in tqdm(dataset):
    price = datapoint["price"]
    if price and price > max_price:
        max_datapoint = datapoint
        max_price = price

print(f"The most expensive wine is from {max_datapoint['winery']} and it costs ${max_price:,.2f}")
print(max_datapoint["description"])

## Remove duplicates first

Ed removed duplicates after parsing, by title and then by full text.

For wine we do it before parsing, on the review text itself. This dataset is two scrapes of the same magazine joined together, and they overlap heavily. The same review often appears twice: once with a title (the newer scrape) and once without.

So we sort the rows that have a title to the front, then keep the first copy of each review. That way, when a review appears twice, we keep the copy with the title.

Expect about 111,000 duplicates to go, leaving 169,430.

In [ ]:
rows = list(dataset)
rows.sort(key=lambda row: row["title"] is None)

seen = set()
rows = [row for row in tqdm(rows) if not (row["description"] in seen or seen.add(row["description"]))]

del seen
print(f"After deduplication, we have {len(rows):,} reviews from {len(dataset):,}")

## Parse into Items

The rules are in `pricer/parser.py`. A wine is kept if:

- it has a price between \$4 and \$200 (Ed used \$0.50 to \$999 for products)
- it has a winery, a variety and a country
- its review is at least 100 characters long

For each wine we build a `full` text with one detail per line, then the review. About half the wines have no title, so the parser builds one from the winery, designation, variety and region.

Expect 154,408 items.

In [ ]:
items = [parse(row) for row in tqdm(rows)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(rows):,} reviews")

In [ ]:
items[1]

In [ ]:
print(items[1].full)

In [ ]:
# And one where the parser had to build the title

print(items[-2].full)

In [ ]:
# Plot the distribution of lengths

lengths = [len(item.full) for item in items]

plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 1000, 25))
plt.show()

In [ ]:
# Plot the distribution of prices

prices = [item.price for item in items]

plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 205, 5))
plt.show()

In [ ]:
# How many in each category? For wine, the category is the country

category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

## Choose our 22,000 wines

The price chart above is lopsided: most wines are cheap, with an average near \$33 and only 15% above \$50. If we picked 22,000 at random, the model would see very few expensive wines and would learn to guess low.

Ed fixed the same problem by giving each product a weight based on its price, then sampling with those weights so that expensive products are picked more often. He used `w = p**2`.

For wine I use the much gentler `w = p**0.5` (the square root). It raises the average from \$33 to about \$41, and gives the model roughly 1,000 wines above \$100 to learn from, about twice as many as a plain random sample would.

Why not go further? The test set is cut from this same sample. The closer the sample stays to real wine prices, the more the errors we measure later reflect how the model would do on real wines. For comparison:

| Weighting | Average price | Share above \$50 |
|---|---|---|
| none | \$33 | 15% |
| `p**0.5` (chosen) | \$41 | 25% |
| `p` | \$51 | 36% |
| `p**2` (Ed's) | \$65 | 56% |

Try changing `w = p**0.5` to `w = p` or `w = p**2` and re-running the cells below to see the effect.

In [ ]:
random.seed(42)
random.shuffle(items)

In [ ]:
np.random.seed(42)

SIZE = 22_000

prices = np.array([it.price for it in items], dtype=float)
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p ** 0.5

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 205, 5))
plt.show()

In [ ]:
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

## Is there a simple pattern?

Ed checked whether price follows text length or weight. Wine has no weight, but it has the reviewer's score in points (80 to 100), so we check that instead.

In [ ]:
# How does the price vary with the character count?

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')
plt.show()

In [ ]:
# How does the price vary with the points?

points = [item.points for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(points, prices, s=0.2, color="darkorange")
plt.xlabel('Points')
plt.ylabel('Price')
plt.title('Is there a simple correlation with points?')
plt.show()

## Now push this dataset to the HuggingFace Hub

20,000 to train, 1,000 to validate and 1,000 to test - the same sizes as Ed's lite dataset.

The sample was drawn in a random order, so slicing it gives a random split.

In [ ]:
from huggingface_hub.utils import reset_sessions
reset_sessions()


In [ ]:
lite = f"{username}/wines_raw_lite"

train = sample[:20_000]
val = sample[20_000:21_000]
test = sample[21_000:]

Item.push_to_hub(lite, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{lite}")